# 계산 그래프
- 계산과정을 유향 비순환 그래프로 표현
    - 노드가 값 또는 국소 연산
    - 간선이 값의 의존 관계 표기
- 단순한 국소적 계산 결과를 전달하여 복잡한 전체 계산
    - 입력 노드부터 연산 노드들을 위상 순서대로 평가 하여 출력 노드의 값 계산
    - 중간 결과 저장 가능
- 미분 가능한 경우 선택한 스칼라 출력에 대한 각 노드에 대한 민감도 계산 가능
    - 연쇄 법칙에 따른 역방향 민감도 누적

## 연쇄 법칙
- 합성 함수의 미분을 구성 함수 미분의 곱으로 나타낼 수 있음
- $$\frac{\partial f}{\partial x} =\frac{\partial f}{\partial t} \times \frac{\partial t}{\partial x}$$
- 전체 출력의 입력에 대한 민감도를 중간 변수에 대한 국소 민감도들의 곱으로 분해 가능
- 계산 그래프에서 각 노드의 국소 미분을 조합하여 전체 민감도 계산

## 역전파

- 미분 가능한 계산 그래프에서 선택한 스칼라 출력에 대한 각 노드의 민감도를 역방향으로 누적하는 알고리즘
  - 현재 순전파에서 계산된 각 노드값에서의 기울기
  - 간선을 거꾸로 따라가며 국소 미분값 곱셈
  - 연쇄법칙으로 여러 경로에서 전달된 민감도 합산|

- 출력 노드에서 시작
  - $$\frac{\partial s}{\partial s} = 1$$

- 출력 노드 $s$에 대한 각 노드 $v$의 민감도
  - $$\bar v = \frac{\partial s}{\partial v}$$

- 딥러닝에서는 손실 $L$에 대한 노드별 민감도

    - $\theta$에 대한 기울기
    - $$\frac{\partial L}{\partial \theta}$$


In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from tqdm import tqdm

import numpy as np
import matplotlib.pyplot as plt

from b1.func import sigmoid, softmax, cee
from b1.func import num_grad as _num_grad

### 덧셈
- 상류의 미분을 그대로 출력
- $$z = x + y$$
- $$\frac{\partial z}{\partial x} = 1, \;  \frac{\partial z}{\partial y} = 1$$

In [ ]:
class AddLayer:
    def __init__(self):
        pass

    def forward(self, x, y):
        out = x + y
        return out

    def backward(self, dout):
        dx = dout * 1
        dy = dout * 1
        return dx, dy

### 곱셈
- 상류의 미분에 하류 신호를 서로 바꾸어 출력
    - x, y 입력일때, x쪽에 y, y쪽에 x 를 곱함 
- $$z = x \times y$$
- $$\frac{\partial z}{\partial x} = y, \; \frac{\partial z}{\partial y} = x$$

In [ ]:
class MulLayer:
    def __init__(self):
        self.x = None
        self.y = None

    def forward(self, x, y):
        self.x = x
        self.y = y
        out = x * y
        return out

    def backward(self, dout):
        dx = dout * self.y  # 상류 출력에 바꾸어 곱함
        dy = dout * self.x
        return dx, dy

### ReLU
- 양수일때는 상류 미분을 그대로, 음수일때는 0 전달
- $$y = \begin{cases} x & (x > 0) \\ 0 & (x \leq 0) \\ \end{cases}$$
- $$\frac{\partial y}{\partial x} = \begin{cases} 1 & (x > 0) \\ 0 & (x \leq 0) \\ \end{cases}$$

In [ ]:
class Relu:
    def __init__(self):
        self.mask = None

    def forward(self, x):
        self.mask = (x <= 0)
        out = x.copy()
        out[self.mask] = 0
        return out

    def backward(self, dout):
        dout[self.mask] = 0
        dx = dout
        return dx

### sigmoid
- $y(1-y)$ 을 상류 미분에 곱하여 전달
- $$y = \frac{1}{1+\exp(-x)}$$
- $$\frac{\partial y}{\partial x} = y^2(\exp(-x)) = y(1-y)$$

In [3]:
class Sigmoid:
    def __init__(self):
        self.out = None

    def forward(self, x):
        self.out = 1 / (1 + np.exp(-x))
        return self.out

    def backward(self, dout):
        dx = dout * self.out * (1 - self.out)
        return dx

### affine
- 어파인 변환 
    - 선형 변환 후 평행이동
    - 활성화 함수 적용 전까지 뉴런 연결

- $$Y = XW + b$$
- $$dX = dYW^T$$
- $$dW = X^TdY$$
- $$db = \sum_{n=1}^{N}dY_n$$

#### 선형 변환
- 요소별 전달로 바꾼뒤 유도하면 다음을 얻을 수 있음 
- $$Y = XW$$
- $$\frac{\partial Y_{nh}}{\partial X_{nd}} = W_{dh}$$
- $$\frac{\partial L}{\partial X_{nd}} = \sum {\frac{\partial L}{\partial Y_{nh}}\frac{\partial Y_{nh}}{\partial X_{nd}} }= \sum_h Y_{nh}W_{dh}$$
- $$\underset{N\times D}{dX} = \underset{(N\times H)(H\times D)}{dYW^T}$$

In [4]:
class Affine:
    def __init__(self, W, b):
        self.W = W
        self.b = b
        self.x = None
        self.dW = None
        self.db = None

    def forward(self, x):
        self.x = x
        out = np.dot(self.x, self.W) + self.b
        return out

    def backward(self, dout):
        dx = np.dot(dout, self.W.T)
        self.dW = np.dot(self.x.T, dout)
        self.db = np.sum(dout, axis=0)
        return dx

### Softmax with Loss
- softmax와 cross entropy error를 하나로 묶은 계층
    - 순전파에서는 점수 벡터를 확률분포로 바꾸고, 정답 라벨에 대한 평균 손실을 계산
    - 역전파에서는 예측 확률과 정답 라벨의 차이를 배치 크기로 나누어 전달
    - 둘을 합치면 미분이 쉬움
- 예측 확률과 정답 라벨의 차이 전달

- softmax $$y_{nk} = \frac{\exp(x_{nk})}{\sum_{i=1}^{K}\exp(x_{ni})}$$

- cee $$L = -\frac{1}{N}\sum_{n=1}^{N}\sum_{k=1}^{K} t_{nk}\log y_{nk}$$

- $$\frac{\partial L}{\partial x_{nk}} = \frac{1}{N}(y_{nk} - t_{nk}) \to d\mathbf x = \frac{1}{N}(\mathbf y - \mathbf t)$$

In [ ]:
class SoftmaxWithLoss:
    def __init__(self):
        self.loss = None
        self.y = None  
        self.t = None  

    def forward(self, x, t):
        self.t = t
        self.y = softmax(x)
        self.loss = cee(self.y, self.t)
        return self.loss

    def backward(self, dout=1):
        batch_size = self.t.shape[0]
        dx = (self.y - self.t) / batch_size
        return dx